# A1: Logic and lambda calculus

**Simon Dobnik and Robin Cooper**

The lab is an exploration and learning exercise to be done in a group and also in discussion with the teachers and other students.

Before starting, please read the instructions on how to work in groups on Canvas.

Write all your answers and the code in the appropriate boxes below.

**On using generative AI for this assignment:** Generative AI has been trained to generate logic expressions. There is floruishing ongoing research how to bridge formal and distributed models to bring improve to the latter known as *neuro-symbolic AI*. Since generative AI can give you direct solutions to this assignment (which may be correct or incorrect), defeats the purpose of doingthe assignment which is to familiarise yourself with writing logic-based semantic grammars. Using generative AI is thus identical to copying an asnwer from someone else. For this reason, **generative AI should not be used** to complete this assignment.

**Running the notebook:** The notebook requires a few dependencies, most notably NLTK which is widely available. However, to draw graphical trees a few other libraries are required but which some students found hard to install on their own computers in the previous years, depedning on what configurations they were using. However, drawing trees graphically is not necessary to work on the assignment as you can also use the bracketing notation in case you cannot get them work. Atternatively, you can run the notebook on the mlt server trhough an ssh tunnel where these dependencies should be available for install.

We recommend that you create a virtual environment (either with virtualenv or conda), install Jupyter Lab in it and all dependencies used in the assignment. To run Jupyter Lab later within your environment, run the following command:

```python -m ipykernel install --user --name=my-virtualenv-name```,

where you replace `my-virtualenv-name` with the name of your created environment.
Once in Jupyter, choose the kernel with the name of your environment. You can do it by either (i) using the drop-down menu in the top right corner or (ii) going to the top menu -> Kernel -> Change Kernel.

**Getting help:** We encourage you to use Canvas discussions to post questions and interact with teachers and also each other. Provide youseful tips, but of course do not reveal the exact answer across the groups as each group should should work out their own solutions. Remember that in most cases there is also not a single correct answer.

## Translating English to logic and evaluating logic in a model

In [ ]:
!pip install nltk IPython

In [ ]:
# This task needs NLTK and Jupyter Lab (IPython package).
import nltk
from utils import display_latex, display_translation, display_tree, display, Markdown

read_expr = nltk.sem.Expression.fromstring

### 1. Propositional logic
Translate the following sentences into **propositional logic** and verify that they parse with Expression.fromstring() (`read_expr` variable in the cell above). Provide a key which shows how the propositional variables in your translation correspond to expressions of English. Briefly discuss any difficulties you encounter. (By difficulties we mean cases where the semantics of English expressions cannot be expressed to the same degree by the semantics of your logic representations, i.e. they do not mean the same). **[5 + 1 marks]**

In [ ]:
keys = {
    "alexPlaysPiano": "Alex plays the piano",
    "alexIsSmart": "Alex is smart",
    "alexIsMusical": "Alex is musical",
    "lydiaIsHappy": "Lydia is happy",
    "georgePlaysPiano": "George plays the piano",
    "georgeIsMusical": "George is musical"
}

propositions = {
    "If Alex plays the piano, she is smart.":
        read_expr('alexPlaysPiano -> alexIsSmart'),

    "Alex is both smart and musical.":
        read_expr('alexIsSmart & alexIsMusical'),

    "If Alex is not smart, Lydia is not happy.":
        read_expr('!alexIsSmart -> !lydiaIsHappy'),

    "If Alex or George plays the piano, they are musical.":
        read_expr('(alexPlaysPiano | georgePlaysPiano) -> (alexIsMusical & georgeIsMusical)'),

    "George plays the piano.":
        read_expr('georgePlaysPiano'),
}

for text, semrep in propositions.items():
    display_translation(text, semrep)

*Difficulties encountered:*

_"If Alex plays the piano, she is smart."_\
We are assuming the "she" is referring to Alex.

_"If Alex or George plays the piano, they are musical."_\
Not sure if should keep the "they" as a single key or to split them.

### 2. Valuation of Propositional logic

Imagine that we observe a world where
- (i) Alex does not play the piano,
- (ii) Alex and Lydia are smart and musical,
- (iii) George is not musical,
- (iv) Lydia is happy,
- (v) George plays the piano.

Translate this informal description of the world into a model by appropriately defining an evaluation function and evaluate the formulae from Question 1 in this model. Briefly comment the answers you get. **[5 + 1 marks]**.

In [ ]:
# answers here
val = nltk.Valuation([
    ('alexPlaysPiano', False),
    ('alexIsSmart', True),
    ('alexIsMusical', True),
    ('lydiaIsSmart', True),
    ('lydiaIsMusical', True),
    ('georgeIsMusical', False),
    ('lydiaIsHappy', True),
    ('georgePlaysPiano', True),
])

dom = set()
assignment = nltk.Assignment(dom)
model = nltk.Model(dom, val)

for text, semrep in propositions.items():
    result = model.evaluate(str(semrep), assignment)
    print(f"{semrep} — {result}")

*Comments:*
1. True : The statement holds true for all cases where "Alex doesnt play piano" OR "Alex is smart". From our grammar we know that Alex is smart. Hence it evaluates True for all cases. 
2. True :  Both "Alex is smart" and "Alex is musical" are true in our model. Since AND is only true when both sides hold, the result is True.
3. True : According to our model, -alexIsSmart and -lydiaIsHappy is False. Hence the implication is True since both statements are False.
4. False : alexPlaysPiano OR georgePlaysPiano gives us True. alexIsMusical AND georgeIsMusical gives us False. Either one of them playing piano, does not directly imply both of them are musical. Hence we get False.
5. True : In our world, georgePlaysPiano is True and there is no statement in our world, that contradicts this. Hence the result is True.

### 3. Predicate logic *without quantifiers*

Translate the following sentences into predicate-argument formulae of First Order Logic and verify that they parse with `Expression.fromstring()`. Briefly discuss any difficulties you encounter. **[4 + 1 marks]**

In [ ]:
sentences1 = {
    "Lydia likes George but Lydia doesn't like Alex":
        read_expr(r'likes(lydia, george) & !likes(lydia, alex)'),

    "Lydia likes herself and so does George":
        read_expr(r'likes(lydia, lydia) & likes(george, george ) '),

    "Charlie is an English pianist who plays a sonata":
        read_expr(r'english(charlie) & pianist(charlie) & plays(charlie, sonata)'),

    "Lydia and George admire each other":
        read_expr(r'admires(lydia,george) & admires(george,lydia)'),
}

for text, semrep in sentences1.items():
    display_translation(text, semrep)


*Difficulties encountered:*

_"Lydia likes George but Lydia doesn't like Alex"_\
I treated the word 'but' as simple AND(&) when translating, ignoring the contrast nuance."

_"Lydia likes herself and so does George"_\
The sentence is ambiguous: it can mean either "Lydia likes herself and George likes himself" or "Lydia likes herself and George likes her".

_"Charlie is an English pianist who plays a sonata"_\
Not sure if need a separate predicate for sonata — `music(sonata)`.

_"Lydia and George admire each other"_\
Not sure if we can use equivalence (<->) since the admiration happens both ways.

### 4. First order logic with quantifiers

Translate the following sentences into quantified formulas of First Order Logic and verify that they parse with `Expression.fromstring()`. Briefly discuss any difficulties you encounter. **[4 + 1 marks]**

In [ ]:
sentences2 = {
    "Charlie knows a woman who likes George":
        read_expr('exists x.(woman(x) & knows(charlie, x) & likes(x, george))'),

    "George admires everybody and Lydia admires nobody":
        read_expr('all x.(person(x) -> admires(george, x)) & !exists y.(person(y) & admires(lydia, y))'),

    "2 - George admires everybody and Lydia admires nobody":
        read_expr('all x.(admires(george, x)) & !exists y.(admires(lydia, y))'),

    "Nobody admires everybody":
        read_expr('!exist x.( all y.(admires(x, y)) )'),

    "Exactly one musician plays everything Alex wrote":
        read_expr('exists x.(musician(x) & all y.(wrote(alex, y) -> plays(x, y)) & all z.((musician(z) & all y.(wrote(alex, y) -> plays(z, y))) -> z = x))'),
}

for text, semrep in sentences2.items():
    display_translation(text, semrep)

*Difficulties encountered:*

_"Charlie knows a woman who likes George"_\
It was tricky figuring out how to represent 'a woman who likes George' correctly and connect it properly to 'Charlie knows' in first order logic.

_"George admires everybody and Lydia admires nobody"_\
I initially had some difficulty with the correct syntax in NLTK. In particular, I used the wrong syntax to express negation, which resulted in an error because NLTK requires '!' as the negation operator.

_"Nobody admires everybody"_\
We had to think in reverse, to build "Somebody admires everybody" first, then negate it. It was also confusing why $\neg \forall x. \forall y. admires(x, y)$ doesn't mean the same thing.      

### 5. Valuation of first order logic

We observe a world with entities Lydia, George, Alex, Charlie and Bertie, sonata, etude, prelude, waltz, scherzo.

1. Lydia likes Lydia, George, Alex and Charlie. George likes Lydia, Bertie and George. Alex likes Alex. Charlie likes Lydia, George, Alex, Charlie and Bertie. Bertie likes Alex.
2. Lydia, George, Alex, Charlie and Bertie are English.
3. Charlie and Bertie are pianists.
4. Charlie plays a sonata, an etude and a waltz. Bertie plays a waltz and a scherzo. Lydia plays an etude, a prelude and a waltz.
5. Lydia admires Lydia, Charlie and Bertie. George admires Lydia, George, Alex, Charlie and Bertie. Alex admires Lydia, Alex and Bertie. Charlie admires George and Bertie. Bertie admires Lydia, George, Alex, Charlie and Bertie.
6. Lydia knows Lydia, George, Alex, Charlie and Bertie. George knows Lydia, George and Bertie. Alex knows Lydia, Alex and Bertie. Charlie knows George, Charlie and Bertie. Bertie knows Lydia, George, Alex, Charlie and Bertie.
7. Lydia, Alex and Charlie are women.
8. George and Bertie are men.
9. Alex wrote a sonata, an etude an a waltz.
10. Lydia, Alex, Charlie and Bertie are musicians.

Translate this informal description of the world into a model and evaluate the formulae from Questions 3 and 4 in this model. Briefly comment on the answers you get **[3 + 2 marks]**.

In [ ]:
# entities = set(['p', 't', 'e', 'h', 'r', 's', 'u', 'l', 'w', 'c'])
entities = set(['lydia', 'george', 'alex', 'charlie', 'bertie', 'sonata', 'etude', 'prelude', 'waltz', 'scherzo'])
assign = """
    lydia => lydia
    george => george
    alex => alex
    charlie => charlie
    bertie => bertie

    sonata => sonata
    etude => etude
    prelude => prelude
    waltz => waltz
    scherzo => scherzo

    likes => {(lydia, lydia), (lydia, george), (lydia, alex), (lydia, charlie), (george, lydia), (george, bertie), (george, george), (alex, alex), (charlie, lydia), (charlie, george), (charlie, alex), (charlie, charlie), (charlie, bertie), (bertie, alex)}

    english => {lydia, george, alex, charlie, bertie}

    pianist => {charlie, bertie}

    plays => {(charlie, sonata), (charlie, etude), (charlie, waltz), (bertie, waltz), (bertie, scherzo), (lydia, etude), (lydia, prelude), (lydia, waltz)}

    admires => {(lydia, lydia), (lydia, charlie), (lydia, bertie), (george, lydia), (george, george), (george, alex), (george, charlie), (george, bertie), (alex, lydia), (alex, alex), (alex, bertie), (charlie, george), (charlie, bertie), (bertie, lydia), (bertie, george), (bertie, alex), (bertie, charlie), (bertie, bertie)}
    
    knows => {(lydia,lydia),(lydia,george),(lydia,alex),(lydia,charlie),(lydia,bertie),(george,lydia),(george,george),(george,bertie),(alex,lydia),(alex,alex),(alex,bertie),(charlie,george),(charlie,charlie),(charlie,bertie),(bertie,lydia),(bertie,george),(bertie,alex),(bertie,charlie),(bertie,bertie)}
    
    woman => {lydia,alex,charlie}
    
    man => {george,bertie}
    
    wrote => {(alex,sonata),(alex,etude),(alex,waltz)}
    
    musician => {lydia,alex,charlie,bertie}
"""

# 1. Lydia likes Lydia, George, Alex and Charlie. George likes Lydia, Bertie and George. Alex likes Alex. Charlie likes Lydia, George, Alex, Charlie and Bertie. Bertie likes Alex.
# 2. Lydia, George, Alex, Charlie and Bertie are English.
# 3. Charlie and Bertie are pianists.
# 4. Charlie plays a sonata, an etude and a waltz. Bertie plays a waltz and a scherzo. Lydia plays an etude, a prelude and a waltz.
# 5. Lydia admires Lydia, Charlie and Bertie. George admires Lydia, George, Alex, Charlie and Bertie. Alex admires Lydia, Alex and Bertie. Charlie admires George and Bertie. Bertie admires Lydia, George, Alex, Charlie and Bertie.
#6. Lydia knows Lydia, George, Alex, Charlie and Bertie. George knows Lydia, George and Bertie. Alex knows Lydia, Alex and Bertie. Charlie knows George, Charlie and Bertie. Bertie knows Lydia, George, Alex, Charlie and Bertie.
#7. Lydia, Alex and Charlie are women.
#8. George and Bertie are men.
#9. Alex wrote a sonata, an etude an a waltz.
#10. Lydia, Alex, Charlie and Bertie are musicians.

val2 = nltk.Valuation.fromstring(assign)

g2 = nltk.Assignment(entities)
m2 = nltk.Model(entities, val2)

# sentences from question 3
print('\n===== sentences from question 3 =====')
for text, semrep in sentences1.items():
    print(m2.evaluate(str(semrep), g2))
    display_latex(semrep)
    display(Markdown('----'))

# sentences from question 4
print('\n===== sentences from question 4 =====')
for text, semrep in sentences2.items():
    print(m2.evaluate(str(semrep), g2))
    display_latex(semrep)
    display(Markdown('----'))


*Comments on the answers for Sentences from Question 3:*

1. False - Our model holds TRUE for Lydia liking George. But Lydia not liking Alex fails as our model holds True for Lydia liking Alex. With one part of the conjuction being False, the whole formula becomes False.
2. True - We have assumed both likes their own selves, which is True in our world. Hence the whole statement is True
3. True - Based on our world, Charlie is English, a pianist and plays the sonata. With every part of the conjunction being True, the whole formula is True.
4. False - In our model, Lydia does not admire George. Hence the whole statement returns False due to conjunction.

*Comments on the answers for Sentences from Question 4:*

1. 
2. 
3. 
4. 


## Lambda calculus

In [ ]:
from nltk.grammar import FeatureGrammar

### 6. Function application and $\beta$-reduction
In the following examples some code has been deleted and replaced with `<????>`. What has been deleted? Verify that your answer is correct. **[4 marks]**

In [ ]:
e1 = read_expr(r'\x.like(x, rob)')
e2 = read_expr(r'pip')
e3 = nltk.sem.ApplicationExpression(e1, e2)
display_latex(e3.simplify())
# with result like(pip,rob).
display_latex(read_expr(r"like(pip,rob)"))

e1 = read_expr(r'\P.P(pip)')
e2 = read_expr(r'\x.play(x,scherzo)')
e3 = nltk.sem.ApplicationExpression(e1, e2)
display_latex(e3.simplify())
# with result play(pip,scherzo).
display_latex(read_expr(r"play(pip,scherzo)"))

e1 = read_expr(r'\P.exists x.(woman(x) & P(x))')
e2 = read_expr(r'\x.play(x,etude)')
e3 = nltk.sem.ApplicationExpression(e1, e2)
display_latex(e3.simplify())
# with result exists x.(woman(x) & play(x,etude)).
display_latex(read_expr(r"exists x.(woman(x) & play(x,etude))"))

e1 = read_expr(r'\Y x.Y(\z.like(x,z))')
e2 = read_expr(r'\P.all x. (musician(x) -> P(x))')
e3 = nltk.sem.ApplicationExpression(e1, e2)
display_latex(e3.simplify())
# with result \x.all z2.(musician(z2) -> like(x,z2)).
display_latex(read_expr(r"\x.all z2.(musician(z2) -> like(x,z2))"))

### 7. Extending the grammar

Extend the grammar simple_sem.fcfg that comes with NLTK `(~/nltk_data/grammars/book_grammars/)` so that it will cover the following sentences:

- no man gives a bone to a dog **[4 marks]**
- no man gives a bone to the dog **[4 marks]**
- a boy and a girl chased every dog **[2 marks]**
- every dog chased a boy and a girl **[2 marks]**
- a brown cat chases a white dog **[4 marks]**

The last example includes adjectives. Several different kinds of adjectives are discussed in the literature [(cf. Kennedy, 2012)](http://semantics.uchicago.edu/kennedy/docs/routledge.pdf). In this example we have an intersective adjective. The denotiation we want for "brown cat" is a a set that we get by intersecting the set of individuals that are brown and the set of individuals that are cats.

C. Kennedy. Adjectives. In G. Russell, editor, The Routledge Companion to Philosophy of Language, chapter 3.3, pages 328–341. Routledge, 2012.

The original grammar is included in the code below as a string.

In [ ]:
fcfg_string_orginal = r"""
% start S
############################
# Grammar Rules
#############################

S[SEM = <?subj(?vp)>] -> NP[NUM=?n,SEM=?subj] VP[NUM=?n,SEM=?vp]

NP[NUM=?n,SEM=<?det(?nom)> ] -> Det[NUM=?n,SEM=?det]  Nom[NUM=?n,SEM=?nom]
NP[LOC=?l,NUM=?n,SEM=?np] -> PropN[LOC=?l,NUM=?n,SEM=?np]

Nom[NUM=?n,SEM=?nom] -> N[NUM=?n,SEM=?nom]

VP[NUM=?n,SEM=?v] -> IV[NUM=?n,SEM=?v]
VP[NUM=?n,SEM=<?v(?obj)>] -> TV[NUM=?n,SEM=?v] NP[SEM=?obj]
VP[NUM=?n,SEM=<?v(?obj,?pp)>] -> DTV[NUM=?n,SEM=?v] NP[SEM=?obj] PP[+TO,SEM=?pp]

PP[+TO, SEM=?np] -> P[+TO] NP[SEM=?np]

#############################
# Lexical Rules
#############################

PropN[-LOC,NUM=sg,SEM=<\P.P(angus)>] -> 'Angus'
PropN[-LOC,NUM=sg,SEM=<\P.P(cyril)>] -> 'Cyril'
PropN[-LOC,NUM=sg,SEM=<\P.P(irene)>] -> 'Irene'

Det[NUM=sg,SEM=<\P Q.all x.(P(x) -> Q(x))>] -> 'every'
Det[NUM=pl,SEM=<\P Q.all x.(P(x) -> Q(x))>] -> 'all'
Det[SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'some'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'a'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'an'

N[NUM=sg,SEM=<\x.man(x)>] -> 'man'
N[NUM=sg,SEM=<\x.girl(x)>] -> 'girl'
N[NUM=sg,SEM=<\x.boy(x)>] -> 'boy'
N[NUM=sg,SEM=<\x.bone(x)>] -> 'bone'
N[NUM=sg,SEM=<\x.ankle(x)>] -> 'ankle'
N[NUM=sg,SEM=<\x.dog(x)>] -> 'dog'
N[NUM=pl,SEM=<\x.dog(x)>] -> 'dogs'

IV[NUM=sg,SEM=<\x.bark(x)>,TNS=pres] -> 'barks'
IV[NUM=pl,SEM=<\x.bark(x)>,TNS=pres] -> 'bark'
IV[NUM=sg,SEM=<\x.walk(x)>,TNS=pres] -> 'walks'
IV[NUM=pl,SEM=<\x.walk(x)>,TNS=pres] -> 'walk'
TV[NUM=sg,SEM=<\X x.X(\ y.chase(x,y))>,TNS=pres] -> 'chases'
TV[NUM=pl,SEM=<\X x.X(\ y.chase(x,y))>,TNS=pres] -> 'chase'
TV[NUM=sg,SEM=<\X x.X(\ y.see(x,y))>,TNS=pres] -> 'sees'
TV[NUM=pl,SEM=<\X x.X(\ y.see(x,y))>,TNS=pres] -> 'see'
TV[NUM=sg,SEM=<\X x.X(\ y.bite(x,y))>,TNS=pres] -> 'bites'
TV[NUM=pl,SEM=<\X x.X(\ y.bite(x,y))>,TNS=pres] -> 'bite'
DTV[NUM=sg,SEM=<\Y X x.X(\z.Y(\y.give(x,y,z)))>,TNS=pres] -> 'gives'
DTV[NUM=pl,SEM=<\Y X x.X(\z.Y(\y.give(x,y,z)))>,TNS=pres] -> 'give'

P[+to] -> 'to'
"""

Write your extension of this grammar here:

In [ ]:
fcfg_string = fcfg_string_orginal + r"""
## Your answers here
# Det[???] -> ???
# TV[???] -> ???
# TV[???] -> ???
# CONJ -> ???
# NP[???] -> NP[???] CONJ NP[???]
# N[???] -> ???
# ADJ[???] -> ???
# NP[???] -> Det[???] ADJ[???] Nom[???]
"""

# Load `fcfg_string` as a feature grammar:
syntax = FeatureGrammar.fromstring(fcfg_string)

Run the code below without errors:

In [ ]:
# comment out sentences if you couldn't find an answer for them
sentences = [
    'no man gives a bone to a dog',
    'no man gives a bone to the dog',
    'a boy and a girl chased every dog',
    'every dog chased a boy and a girl',
    'a brown cat chases a white dog',
]
for results in nltk.interpret_sents(sentences, syntax):
    for (synrep, semrep) in results:
        display(Markdown('----'))
        display_latex(semrep)  # prints the SEM feature of a tree
        display_tree(synrep)  # show the parse tree

If you are working with iPython which is also running behind Jupyter notebooks and you are changing grammars and want to rerun a new version without restarting you may find `nltk.data.clear_cache()` useful.

## Statement of contribution

Briefly state how many times you have met for discussions, who was present, to what degree each member contributed to the discussion and the final answers you are submitting.

We first read the required literature individually and held a discussion before the first lab session, during which we went through all the questions on the list together. We then met in person during the first lab session, where we completed parts 1 and 2. Other than that, we mainly keep in touch via WhatsApp all the time.

All the team members were active in participation. We split the work equally so that each of us would have a chance to work on each part individually, e.g. part 3 and 4, each of us handle each sentence. We would review and check-in on one another's work when they are done.For part 5 and 6 which are more challenging, we split into pair work.

For parts 5 and 6, which are more challenging and involve larger chunks of content, splitting the work by sentences was not practical. Instead, we worked in pairs and conducted cross-checking to ensure correctness and coherence.

For part 7, 


## Marks

The assignment is marked on a 7-level scale where 4 is sufficient to complete the assignment; 5 is good solid work; 6 is excellent work, covers most of the assignment; and 7: creative work.

This assignment has a total of 47 marks. These translate to grades as follows: 1 = 17% 2 = 34%, 3 = 50%, 4 = 67%, 5 = 75%, 6 = 84%, 7 = 92% where %s are interpreted as lower bounds to achieve that grade.